# Huấn luyện Box Critic trên Google Colab (PointSnap 3D, lớp 3 của Box Verifier)

Chuẩn bị trên máy (xem `docs/box-critic.md`):

1. `pointsnap critic-data --scenes tuning --samples-per-box 20 --out critic-data`, chép `scripts/train_critic_example.py`
   vào `critic-data/`, nén thành `critic-data.zip` (trong zip có thư mục `critic-data/`).
2. Tải `critic-data.zip` lên **Google Drive của bạn** (thư mục gốc *My Drive*). Không chia sẻ công khai: giấy phép nuScenes
   không cho phân phối lại.

Trên Colab: **Runtime → Change runtime type → T4 GPU**, rồi chạy lần lượt từng ô.

In [ ]:
# Lấy dữ liệu từ Google Drive (sẽ hỏi quyền truy cập Drive)
from google.colab import drive
drive.mount('/content/drive')
ZIP = '/content/drive/MyDrive/critic-data.zip'   # đổi nếu để ở thư mục khác
!unzip -oq "$ZIP" -d /content
DATA = '/content/critic-data'
!ls -la $DATA

Không dùng Drive? Bỏ qua ô trên, chạy ô dưới để tải `critic-data.zip` thẳng từ máy (chậm hơn với file lớn).

In [ ]:
# from google.colab import files
# up = files.upload()                      # chọn critic-data.zip
# !unzip -oq critic-data.zip -d /content
# DATA = '/content/critic-data'

In [ ]:
import numpy as np, torch
from pathlib import Path
print('torch', torch.__version__, 'GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'KHÔNG CÓ — bật T4 GPU')
for p in sorted(Path(DATA).glob('*.npz')):
    d = np.load(p)
    print(p.name, d['points'].shape, 'tỉ lệ mẫu sai', round(float(d['is_error'].mean()), 3))

Huấn luyện. Script in độ chính xác và sai số gợi ý dời (trung vị) trên tập kiểm tra mỗi epoch; tăng `--epochs` nếu còn đang tốt lên.

In [ ]:
!python $DATA/train_critic_example.py --data $DATA --epochs 40 --batch 512 --out /content/critic.pt

Kiểm tra file đúng **hợp đồng** PointSnap `logit, delta = model(points (B,256,4), meta (B,20))`, lưu vào Drive và tải về máy.

In [ ]:
m = torch.jit.load('/content/critic.pt', map_location='cpu').eval()
logit, delta = m(torch.zeros(2, 256, 4), torch.zeros(2, 20))
assert logit.shape == (2,) and delta.shape == (2, 7)
!cp /content/critic.pt /content/drive/MyDrive/critic.pt
from google.colab import files
files.download('/content/critic.pt')
print('OK — critic.pt đã lưu vào My Drive và tải về máy. Khai trong guideline:  critic: {model: <đường dẫn>/critic.pt}')